# Prophet Tuning

Prophet's defaults are designed to work well out of the box, but many
time series benefit from tuning.  This notebook covers the key levers:

1. **Changepoint tuning** — controlling trend flexibility
2. **Seasonality modes** — additive vs multiplicative
3. **Custom seasonality** — adding non-standard seasonal periods
4. **Holiday effects** — incorporating known events
5. **Logistic growth** — for saturating trends
6. **Comparing default vs tuned models**

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

from prophet import Prophet

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# Load and prepare airline passengers data
airline = pd.read_csv(DATA_DIR / "airline_passengers.csv")
airline_prophet = airline.rename(columns={
    "Month": "ds",
    "Thousands of Passengers": "y",
})
airline_prophet["ds"] = pd.to_datetime(airline_prophet["ds"])

# Load and prepare alcohol sales data
alcohol = pd.read_csv(DATA_DIR / "Alcohol_Sales.csv")
alcohol_prophet = alcohol.rename(columns={
    "DATE": "ds",
    "S4248SM144NCEN": "y",
})
alcohol_prophet["ds"] = pd.to_datetime(alcohol_prophet["ds"])

# Train/test splits
n_test_air = 24
train_air = airline_prophet.iloc[:-n_test_air].copy()
test_air = airline_prophet.iloc[-n_test_air:].copy()

n_test_alc = 36
train_alc = alcohol_prophet.iloc[:-n_test_alc].copy()
test_alc = alcohol_prophet.iloc[-n_test_alc:].copy()

print(f"Airline — Train: {len(train_air)}, Test: {len(test_air)}")
print(f"Alcohol — Train: {len(train_alc)}, Test: {len(test_alc)}")

In [ ]:
def evaluate_prophet(model, train_df, test_df, freq="ME", label="Model"):
    """Fit a Prophet model, forecast, and report accuracy metrics.

    Returns the forecast DataFrame for the test period.
    """
    model.fit(train_df)
    future = model.make_future_dataframe(periods=len(test_df), freq=freq)
    forecast = model.predict(future)

    # Match forecast dates to test dates
    forecast_test = forecast.set_index("ds").loc[test_df["ds"].values].reset_index()
    actual = test_df["y"].values
    pred = forecast_test["yhat"].values

    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100

    print(f"{label}")
    print(f"  MAE:  {mae:.2f}  |  RMSE: {rmse:.2f}  |  MAPE: {mape:.2f}%")
    return forecast, forecast_test

---
## 1. Changepoint Tuning

Prophet automatically detects **changepoints** — dates where the trend
slope changes.  Three parameters control this behavior:

| Parameter | Default | Effect |
|-----------|---------|--------|
| `n_changepoints` | 25 | Number of *potential* changepoint locations |
| `changepoint_range` | 0.8 | Fraction of the training history where changepoints can be placed (first 80%) |
| `changepoint_prior_scale` | 0.05 | Controls trend flexibility — **the most important tuning parameter** |

### `changepoint_prior_scale`

This is a regularization parameter:
- **Higher values** (e.g., 0.5) allow more changepoints to be active → more flexible, wiggly trend
- **Lower values** (e.g., 0.001) suppress changepoints → smoother, more rigid trend
- **Default** (0.05) is a reasonable middle ground

In [ ]:
# Fit default model on alcohol sales and visualize changepoints
model_default = Prophet()
model_default.fit(train_alc)
future_alc = model_default.make_future_dataframe(periods=n_test_alc, freq="MS")
forecast_default = model_default.predict(future_alc)

fig = model_default.plot(forecast_default)
# Add changepoints to the plot
a = model_default.add_changepoints_to_plot(fig.gca())
plt.title("Alcohol Sales — Default Prophet with Changepoints")
plt.ylabel("Millions of Dollars")
plt.tight_layout()
plt.show()

print("Red vertical lines = detected changepoints.")
print("The dashed red line shows the trend slope changes.")

In [ ]:
# Compare different changepoint_prior_scale values
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
scales = [0.001, 0.05, 0.2, 0.5]

for ax, scale in zip(axes.flat, scales):
    m = Prophet(changepoint_prior_scale=scale)
    m.fit(train_alc)
    future = m.make_future_dataframe(periods=n_test_alc, freq="MS")
    fc = m.predict(future)

    ax.plot(train_alc["ds"], train_alc["y"], "k.", markersize=2, alpha=0.5)
    ax.plot(fc["ds"], fc["trend"], color="tab:red", linewidth=2, label="Trend")
    ax.plot(fc["ds"], fc["yhat"], color="tab:blue", alpha=0.5, label="$\\hat{y}$")
    ax.set_title(f"changepoint_prior_scale = {scale}")
    ax.legend(fontsize=8)

plt.suptitle("Effect of changepoint_prior_scale on Trend Flexibility", fontsize=14)
plt.tight_layout()
plt.show()

print("0.001: nearly straight line (underfitting trend changes)")
print("0.05:  default — moderate flexibility")
print("0.5:   very flexible — may overfit to noise in the trend")

In [ ]:
# Quantitative comparison of changepoint_prior_scale values
print("Alcohol Sales — Effect of changepoint_prior_scale")
print(f"{'Scale':<10s} {'MAE':>8s} {'RMSE':>8s} {'MAPE':>8s}")
print("-" * 38)

for scale in [0.001, 0.01, 0.05, 0.1, 0.2, 0.5]:
    m = Prophet(changepoint_prior_scale=scale)
    m.fit(train_alc)
    future = m.make_future_dataframe(periods=n_test_alc, freq="MS")
    fc = m.predict(future)
    fc_test = fc.set_index("ds").loc[test_alc["ds"].values]
    actual = test_alc["y"].values
    pred = fc_test["yhat"].values
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    print(f"{scale:<10.3f} {mae:8.2f} {rmse:8.2f} {mape:7.2f}%")

### `n_changepoints` and `changepoint_range`

- `n_changepoints` sets the number of *potential* changepoint locations (uniformly
  spaced).  Most will have near-zero rate changes due to regularization.
- `changepoint_range` determines how far into the training data changepoints
  can be placed.  Default 0.8 means no changepoints in the last 20% — this
  prevents overfitting at the end of the series but means trend changes near
  the end of training will not be captured.

In [ ]:
# Compare changepoint_range values
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, cr in zip(axes, [0.8, 0.95]):
    m = Prophet(changepoint_range=cr)
    m.fit(train_alc)
    future = m.make_future_dataframe(periods=n_test_alc, freq="MS")
    fc = m.predict(future)

    ax.plot(train_alc["ds"], train_alc["y"], "k.", markersize=2, alpha=0.5)
    ax.plot(test_alc["ds"], test_alc["y"], "g.", markersize=4, label="Test actuals")
    ax.plot(fc["ds"], fc["yhat"], color="tab:blue", alpha=0.6)
    ax.plot(fc["ds"], fc["trend"], color="tab:red", linewidth=2, label="Trend")
    ax.set_title(f"changepoint_range = {cr}")
    ax.legend()

plt.suptitle("Effect of changepoint_range", fontsize=14)
plt.tight_layout()
plt.show()

print("changepoint_range=0.95 allows trend changes closer to the end of training.")

---
## 2. Additive vs Multiplicative Seasonality

By default, Prophet uses **additive** seasonality:
$$y(t) = g(t) + s(t) + h(t) + \varepsilon_t$$

For time series where the seasonal amplitude grows with the level (like
airline passengers), **multiplicative** seasonality is more appropriate:
$$y(t) = g(t) \cdot (1 + s(t)) + h(t) + \varepsilon_t$$

Set this with `Prophet(seasonality_mode='multiplicative')`.

In [ ]:
# Compare additive vs multiplicative on airline passengers
print("Airline Passengers — Additive vs Multiplicative Seasonality\n")

model_add = Prophet(seasonality_mode="additive")
_, fc_add = evaluate_prophet(model_add, train_air, test_air, freq="ME",
                             label="Additive (default)")

print()

model_mult = Prophet(seasonality_mode="multiplicative")
_, fc_mult = evaluate_prophet(model_mult, train_air, test_air, freq="ME",
                              label="Multiplicative")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, fc, title in zip(axes, [fc_add, fc_mult], ["Additive", "Multiplicative"]):
    ax.plot(train_air["ds"], train_air["y"], color="tab:blue", label="Train")
    ax.plot(test_air["ds"], test_air["y"], color="tab:green",
            label="Actual", marker="o", markersize=4)
    ax.plot(fc["ds"], fc["yhat"], color="tab:red",
            label="Forecast", linestyle="--")
    ax.fill_between(fc["ds"], fc["yhat_lower"], fc["yhat_upper"],
                    alpha=0.15, color="tab:red")
    ax.set_title(f"{title} Seasonality")
    ax.legend(fontsize=8)

plt.suptitle("Airline Passengers — Additive vs Multiplicative", fontsize=14)
plt.tight_layout()
plt.show()

print("Multiplicative seasonality produces wider summer peaks at higher")
print("trend levels — matching the data's actual behavior.")

In [ ]:
# Also compare on alcohol sales
print("Alcohol Sales — Additive vs Multiplicative Seasonality\n")

model_add_alc = Prophet(seasonality_mode="additive")
_, fc_add_alc = evaluate_prophet(model_add_alc, train_alc, test_alc, freq="MS",
                                  label="Additive (default)")
print()

model_mult_alc = Prophet(seasonality_mode="multiplicative")
_, fc_mult_alc = evaluate_prophet(model_mult_alc, train_alc, test_alc, freq="MS",
                                   label="Multiplicative")

print()
print("Multiplicative often wins when the seasonal swings grow with the level.")

---
## 3. Custom Seasonality

Prophet auto-detects yearly and weekly seasonality, but you can add
custom seasonal periods with `model.add_seasonality()`.

This is useful for:
- **Monthly** patterns in data with sub-monthly frequency
- **Quarterly** patterns in financial data
- **Bi-annual** patterns (e.g., academic semesters)

In [ ]:
# Add a monthly seasonality to alcohol sales
# This might capture patterns like end-of-month effects
model_custom_season = Prophet(seasonality_mode="multiplicative")
model_custom_season.add_seasonality(
    name="monthly",
    period=30.5,       # approximate days in a month
    fourier_order=5,   # 5 sine/cosine pairs
)

print("Custom seasonality added:")
print(f"  Name:          monthly")
print(f"  Period:        30.5 days")
print(f"  Fourier order: 5")
print()
print("The model now has yearly (auto) + monthly (custom) seasonality.")

In [ ]:
# Evaluate the custom seasonality model
print("Alcohol Sales — Custom Seasonality\n")
_, fc_custom = evaluate_prophet(model_custom_season, train_alc, test_alc,
                                freq="MS", label="Multiplicative + monthly seasonality")

print()
print("Additional seasonality can help if there is a real pattern to capture,")
print("but can also overfit if the extra Fourier terms just model noise.")

### Seasonality Prior Scale

The `seasonality_prior_scale` parameter (default 10) controls how
flexible the seasonal component can be.  Lower values produce smoother
seasonality; higher values allow the seasonal shape to fit more closely
to the data (risking overfitting).

In [ ]:
# Compare different seasonality_prior_scale values
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, sps in zip(axes, [0.1, 10, 50]):
    m = Prophet(seasonality_mode="multiplicative", seasonality_prior_scale=sps)
    m.fit(train_alc)
    future = m.make_future_dataframe(periods=n_test_alc, freq="MS")
    fc = m.predict(future)

    ax.plot(train_alc["ds"], train_alc["y"], "k.", markersize=2, alpha=0.5)
    ax.plot(fc["ds"], fc["yhat"], color="tab:blue", alpha=0.7)
    ax.set_title(f"seasonality_prior_scale = {sps}")

plt.suptitle("Effect of Seasonality Prior Scale", fontsize=14)
plt.tight_layout()
plt.show()

print("Lower values = smoother seasonal pattern.")
print("Higher values = seasonal pattern follows the data more closely.")

---
## 4. Holiday Effects

Prophet can incorporate known holidays and events that disrupt normal
patterns.  Holidays are specified as a DataFrame with columns:
- `holiday`: name of the holiday
- `ds`: date of each occurrence
- `lower_window` / `upper_window`: days before/after to include

In [ ]:
# Define custom holidays for alcohol sales
# Christmas and New Year's likely drive December spikes
christmas = pd.DataFrame({
    "holiday": "christmas",
    "ds": pd.to_datetime([f"{y}-12-25" for y in range(1992, 2020)]),
    "lower_window": -7,   # week before Christmas
    "upper_window": 1,    # day after
})

new_years = pd.DataFrame({
    "holiday": "new_years",
    "ds": pd.to_datetime([f"{y}-01-01" for y in range(1992, 2020)]),
    "lower_window": -1,
    "upper_window": 0,
})

july_4th = pd.DataFrame({
    "holiday": "july_4th",
    "ds": pd.to_datetime([f"{y}-07-04" for y in range(1992, 2020)]),
    "lower_window": -2,
    "upper_window": 1,
})

holidays = pd.concat([christmas, new_years, july_4th], ignore_index=True)

print(f"Total holiday entries: {len(holidays)}")
print(holidays.groupby("holiday").size())
print()
print("Sample entries:")
holidays.head()

In [ ]:
# Fit model with custom holidays
model_holidays = Prophet(
    seasonality_mode="multiplicative",
    holidays=holidays,
)

print("Alcohol Sales — With Custom Holidays\n")
forecast_h, fc_test_h = evaluate_prophet(
    model_holidays, train_alc, test_alc, freq="MS",
    label="Multiplicative + custom holidays",
)

### Country Holidays

Instead of defining holidays manually, Prophet can add all holidays for
a given country automatically.  This uses the `holidays` Python package
under the hood.

In [ ]:
# Use country holidays instead of manual specification
model_country_holidays = Prophet(seasonality_mode="multiplicative")
model_country_holidays.add_country_holidays(country_name="US")

print("Alcohol Sales — With US Country Holidays\n")
forecast_ch, fc_test_ch = evaluate_prophet(
    model_country_holidays, train_alc, test_alc, freq="MS",
    label="Multiplicative + US holidays",
)

print()
print("Country holidays automatically include Thanksgiving, Christmas,")
print("Labor Day, Memorial Day, and many others.")

---
## 5. Logistic Growth (Saturating Trends)

Some time series have a natural **carrying capacity** — a maximum value
the series cannot exceed (e.g., total addressable market, resource limits).

For these, Prophet's **logistic growth** trend is appropriate:

$$
g(t) = \frac{C(t)}{1 + \exp\left(-(k + \mathbf{a}(t)^T \boldsymbol{\delta})(t - (m + \mathbf{a}(t)^T \boldsymbol{\gamma}))\right)}
$$

To use logistic growth:
1. Set `growth='logistic'` in the Prophet constructor
2. Add a `cap` column to the DataFrame (required)
3. Optionally add a `floor` column

In [ ]:
# Demonstrate logistic growth with airline passengers
# Set a hypothetical cap (maximum passengers the industry could handle)
train_air_log = train_air.copy()
train_air_log["cap"] = 700  # hypothetical capacity
train_air_log["floor"] = 0  # passengers cannot be negative

model_logistic = Prophet(growth="logistic")
model_logistic.fit(train_air_log)

# Future DataFrame must also have cap and floor
future_log = model_logistic.make_future_dataframe(periods=n_test_air, freq="ME")
future_log["cap"] = 700
future_log["floor"] = 0
forecast_log = model_logistic.predict(future_log)

fig = model_logistic.plot(forecast_log)
plt.axhline(700, color="red", linestyle="--", alpha=0.5, label="Cap = 700")
plt.title("Airline Passengers — Logistic Growth (cap=700)")
plt.ylabel("Thousands of Passengers")
plt.legend()
plt.tight_layout()
plt.show()

print("The logistic growth trend saturates as it approaches the cap.")
print("This prevents the forecast from growing indefinitely.")

In [ ]:
# Compare linear vs logistic growth component plots
fig = model_logistic.plot_components(forecast_log)
plt.suptitle("Logistic Growth — Components", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Notice how the trend component curves and flattens near the cap,")
print("unlike the straight-line trend of piecewise linear growth.")

---
## 6. Putting It All Together: Default vs Tuned

Let us build a carefully tuned Prophet model for each dataset and
compare against the defaults.

In [ ]:
# Tuned model for airline passengers
print("=" * 60)
print("AIRLINE PASSENGERS — Model Comparison")
print("=" * 60)
print()

# Default
model_default_air = Prophet()
_, fc_default_air = evaluate_prophet(model_default_air, train_air, test_air,
                                     freq="ME", label="1. Default")
print()

# Multiplicative seasonality
model_mult_air = Prophet(seasonality_mode="multiplicative")
_, fc_mult_air = evaluate_prophet(model_mult_air, train_air, test_air,
                                   freq="ME", label="2. Multiplicative seasonality")
print()

# Tuned: multiplicative + adjusted changepoint scale
model_tuned_air = Prophet(
    seasonality_mode="multiplicative",
    changepoint_prior_scale=0.1,
    seasonality_prior_scale=5,
)
_, fc_tuned_air = evaluate_prophet(model_tuned_air, train_air, test_air,
                                    freq="ME", label="3. Tuned (mult + cps=0.1)")

In [ ]:
# Visual comparison of all airline models
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(train_air["ds"], train_air["y"], color="tab:blue", label="Train", alpha=0.6)
ax.plot(test_air["ds"], test_air["y"], color="black", label="Actual",
        marker="o", markersize=5, linewidth=2)

ax.plot(fc_default_air["ds"], fc_default_air["yhat"],
        linestyle="--", label="Default", alpha=0.8)
ax.plot(fc_mult_air["ds"], fc_mult_air["yhat"],
        linestyle="--", label="Multiplicative", alpha=0.8)
ax.plot(fc_tuned_air["ds"], fc_tuned_air["yhat"],
        linestyle="--", label="Tuned", alpha=0.8)

ax.set_title("Airline Passengers — Default vs Tuned Prophet")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Tuned model for alcohol sales
print("=" * 60)
print("ALCOHOL SALES — Model Comparison")
print("=" * 60)
print()

# Default
model_def_alc = Prophet()
_, fc_def_alc = evaluate_prophet(model_def_alc, train_alc, test_alc,
                                  freq="MS", label="1. Default")
print()

# Multiplicative + country holidays
model_tuned_alc = Prophet(
    seasonality_mode="multiplicative",
    changepoint_prior_scale=0.1,
)
model_tuned_alc.add_country_holidays(country_name="US")
_, fc_tuned_alc = evaluate_prophet(model_tuned_alc, train_alc, test_alc,
                                    freq="MS", label="2. Tuned (mult + US holidays + cps=0.1)")

---
## Summary

### Tuning Cheatsheet

| Parameter | Default | What it controls | Increase to... | Decrease to... |
|-----------|---------|-----------------|---------------|---------------|
| `changepoint_prior_scale` | 0.05 | Trend flexibility | More changepoints, wiggly trend | Smoother, rigid trend |
| `n_changepoints` | 25 | Number of potential changepoints | More candidate locations | Fewer candidates |
| `changepoint_range` | 0.8 | Where changepoints are placed | Allow changes later in history | Restrict to earlier history |
| `seasonality_prior_scale` | 10 | Seasonal pattern flexibility | More detailed seasonal shape | Smoother seasonality |
| `seasonality_mode` | `"additive"` | How seasonality combines with trend | Use `"multiplicative"` for growing amplitude | Keep additive for constant amplitude |
| `holidays_prior_scale` | 10 | Holiday effect flexibility | Larger holiday effects | Smaller holiday effects |
| `growth` | `"linear"` | Trend model | Use `"logistic"` for saturation | Keep linear for unbounded growth |

### Key takeaways
- **`seasonality_mode='multiplicative'`** is the single most impactful change for series with growing seasonal amplitude
- **`changepoint_prior_scale`** is the most important continuous tuning parameter
- **Holidays** can help for daily/weekly data; less impactful for monthly aggregates
- **Logistic growth** requires domain knowledge (what is the cap?)

In [ ]:
print("Next notebook: Prophet evaluation — cross-validation, diagnostic metrics,")
print("and head-to-head comparison with SARIMA.")